In [0]:
#pandas schema

import pandas as pd

data = [[6, 'Alice'], [2, 'Bob'], [7, 'Alex']]
users = pd.DataFrame(data, columns=['user_id', 'user_name']).astype({'user_id': 'Int64', 'user_name': 'object'})
data = [[215, 6], [209, 2], [208, 2], [210, 6], [208, 6], [209, 7], [209, 6], [215, 7], [208, 7], [210, 2], [207, 2],
        [210, 7]]
register = pd.DataFrame(data, columns=['contest_id', 'user_id']).astype({'contest_id': 'Int64', 'user_id': 'Int64'})

In [0]:
# to spark dataframe

from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder.getOrCreate()

users_df = spark.createDataFrame(users)
users_df.show(truncate=False)

In [0]:
register_df = spark.createDataFrame(register)
register_df.show(truncate=False)

In [0]:
# solving using spark dataframe

register_df\
        .join(users_df,on=['user_id'],how='inner')\
        .groupBy('contest_id').agg(F.count('user_id').alias('users_count'))\
        .withColumn('percentage',F.round(100*F.col('users_count')/users_df.count(),2))\
        .drop('users_count')\
        .orderBy(F.col('percentage').desc(), 'contest_id')\
        .show()

In [0]:
# solving in spark sql

users_df.createOrReplaceTempView('users')
register_df.createOrReplaceTempView('register')

spark.sql('''
select 
        contest_id, 
        round(100*count(register.user_id)/(select count(user_id) from users),2) as percentage
from register 
inner join users
on register.user_id = users.user_id
group by contest_id
order by percentage desc, contest_id asc;
''').show()